In [ ]:
import torch
import torch.nn as nn
from transformers import XLNetTokenizer
from torch.utils.data import DataLoader

# add src to path to import our modules
import sys
import os
sys.path.append(os.path.abspath('..'))

from src.dataset import ClarityDataset
from src.model import DualHeadXLNet

# check gpu
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"using device: {device}")

In [ ]:
# init tokenizer
tokenizer = XLNetTokenizer.from_pretrained('xlnet-base-cased')

# create small dataset (just 8 samples for overfitting)
full_dataset = ClarityDataset('../data/processed/train.csv', tokenizer)
small_dataset = torch.utils.data.Subset(full_dataset, range(8))

loader = DataLoader(small_dataset, batch_size=4, shuffle=True)

# init model
model = DualHeadXLNet().to(device)
model.train() # set to train mode
print("model initialized")

In [ ]:
# setup optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

# simple loss function (sum of both heads)
criterion = nn.CrossEntropyLoss()

print("starting sanity check (overfitting)...")

for epoch in range(50):
    total_loss = 0
    
    for batch in loader:
        # move to gpu
        ids = batch['input_ids'].to(device)
        mask = batch['attention_mask'].to(device)
        c_labels = batch['clarity_labels'].to(device)
        e_labels = batch['evasion_labels'].to(device)
        
        optimizer.zero_grad()
        
        # forward
        c_logits, e_logits = model(ids, mask)
        
        # calc loss
        loss_c = criterion(c_logits, c_labels)
        loss_e = criterion(e_logits, e_labels)
        loss = loss_c + loss_e
        
        # backward
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
    if epoch % 10 == 0:
        print(f"Epoch {epoch} | Loss: {total_loss:.4f}")

print("Final Loss:", total_loss)
# Should be very close to 0 (e.g., < 0.1)